# **Hands-on Lab: Interactive Visual Analytics with Folium**

En este laboratorio usaremos `Folium` para visualizar los sitios de lanzamiento de SpaceX en un mapa interactivo, marcar los resultados de cada lanzamiento y calcular distancias a proximidades geográficas.

## Objetivos

- **TASK 1:** Marcar todos los sitios de lanzamiento en el mapa
- **TASK 2:** Marcar lanzamientos exitosos y fallidos con colores
- **TASK 3:** Calcular distancias del sitio de lanzamiento a sus proximidades

## Instalación e importación de librerías

In [ ]:
import piplite
await piplite.install(['folium'])
await piplite.install(['pandas'])

In [ ]:
import folium
import pandas as pd
from folium.plugins import MarkerCluster
from folium.plugins import MousePosition
from folium.features import DivIcon
from math import sin, cos, sqrt, atan2, radians

print("Librerías importadas correctamente ✓")

## Carga del dataset

Cargamos el dataset `spacex_launch_geo.csv` que incluye coordenadas de latitud y longitud para cada sitio de lanzamiento.

In [ ]:
from js import fetch
import io

URL = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_geo.csv'
resp = await fetch(URL)
spacex_csv_file = io.BytesIO((await resp.arrayBuffer()).to_py())
spacex_df = pd.read_csv(spacex_csv_file)

print(f"Dataset cargado: {spacex_df.shape[0]} filas x {spacex_df.shape[1]} columnas")
spacex_df.head()

In [ ]:
# Seleccionar columnas relevantes y obtener coordenadas únicas por sitio
spacex_df = spacex_df[['Launch Site', 'Lat', 'Long', 'class']]

launch_sites_df = spacex_df.groupby(['Launch Site'], as_index=False).first()
launch_sites_df = launch_sites_df[['Launch Site', 'Lat', 'Long']]
launch_sites_df

---
## TASK 1: Marcar todos los sitios de lanzamiento en el mapa

Creamos un mapa centrado en el NASA Johnson Space Center (Houston, TX) y añadimos un círculo y etiqueta de texto para cada uno de los 4 sitios de lanzamiento.

In [ ]:
# Coordenadas de NASA Johnson Space Center (centro inicial del mapa)
nasa_coordinate = [29.559684888503615, -95.0830971930759]

# Crear el mapa con zoom que muestre toda la costa este y oeste de EE.UU.
site_map = folium.Map(location=nasa_coordinate, zoom_start=5)

# Añadir un Circle y un Marker de texto para cada sitio de lanzamiento
for index, row in launch_sites_df.iterrows():
    coordinate = [row['Lat'], row['Long']]
    
    # Círculo naranja con popup del nombre del sitio
    folium.Circle(
        coordinate,
        radius=1000,
        color='#d35400',
        fill=True,
        fill_opacity=0.5
    ).add_child(folium.Popup(row['Launch Site'])).add_to(site_map)
    
    # Etiqueta de texto con el nombre del sitio
    folium.map.Marker(
        coordinate,
        icon=DivIcon(
            icon_size=(200, 20),
            icon_anchor=(0, 0),
            html='<div style="font-size: 12px; color:#d35400; font-weight:bold;">%s</div>' % row['Launch Site']
        )
    ).add_to(site_map)

# Mostrar el mapa — TASK 1: tomar screenshot aquí (zoom_start=5, vista global EE.UU.)
site_map

**📸 SCREENSHOT 1:** Captura este mapa con los 4 sitios marcados con círculos naranjas.

Observaciones:
- Los 4 sitios están ubicados cerca del ecuador y junto a la costa
- CCAFS LC-40, CCAFS SLC-40 y KSC LC-39A están en Florida (costa este)
- VAFB SLC-4E está en California (costa oeste)

---
## TASK 2: Marcar lanzamientos exitosos y fallidos con colores

Usamos `MarkerCluster` para agrupar los marcadores. Verde = éxito (class=1), Rojo = fallo (class=0).

In [ ]:
# Función para asignar color según resultado del lanzamiento
def assign_marker_color(launch_outcome):
    if launch_outcome == 1:
        return 'green'
    else:
        return 'red'

# Crear la columna marker_color
spacex_df['marker_color'] = spacex_df['class'].apply(assign_marker_color)

# Verificar resultado
print(f"Total lanzamientos: {len(spacex_df)}")
print(f"Exitosos (verde): {spacex_df[spacex_df['class']==1].shape[0]}")
print(f"Fallidos  (rojo): {spacex_df[spacex_df['class']==0].shape[0]}")
spacex_df.tail(5)

In [ ]:
# Crear un nuevo mapa y añadir el MarkerCluster
site_map2 = folium.Map(location=nasa_coordinate, zoom_start=5)
marker_cluster = MarkerCluster()
site_map2.add_child(marker_cluster)

# Añadir cada lanzamiento como un marcador al cluster
for index, record in spacex_df.iterrows():
    folium.Marker(
        location=[record['Lat'], record['Long']],
        icon=folium.Icon(color='white', icon_color=record['marker_color']),
        popup=folium.Popup(
            f"<b>{record['Launch Site']}</b><br>Resultado: {'Éxito ✓' if record['class']==1 else 'Fallo ✗'}",
            max_width=200
        )
    ).add_to(marker_cluster)

# Mostrar el mapa — TASK 2: tomar screenshot aquí (zoom_start=5, vista global)
site_map2

**📸 SCREENSHOT 2:** Captura este mapa con los clusters de marcadores.

- Haz zoom en cualquier sitio (ej. Florida) para ver los marcadores individuales en verde y rojo
- Captura la vista con zoom en un sitio específico para mostrar los colores claramente

---
## TASK 3: Calcular distancias a las proximidades del sitio de lanzamiento

Usaremos CCAFS SLC-40 como sitio de referencia. Calcularemos la distancia a la costa, una autopista, una vía férrea y la ciudad más cercana usando la fórmula de Haversine.

In [ ]:
# Función de distancia con fórmula de Haversine
def calculate_distance(lat1, lon1, lat2, lon2):
    R = 6373.0  # Radio de la Tierra en km
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = sin(dlat / 2)**2 + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))
    return R * c

In [ ]:
# Coordenadas del sitio de lanzamiento CCAFS SLC-40
launch_site_lat = 28.563197
launch_site_lon = -80.576820

# Coordenadas de proximidades (obtenidas con MousePosition sobre el mapa)
coastline_lat,  coastline_lon  = 28.56367, -80.56763   # Costa Atlántica
highway_lat,    highway_lon    = 28.56321, -80.57079   # Samuel C. Phillips Pkwy
railway_lat,    railway_lon    = 28.57206, -80.58525   # Vía férrea cercana
city_lat,       city_lon       = 28.10621, -80.63731   # Melbourne, FL

# Calcular distancias
dist_coast   = calculate_distance(launch_site_lat, launch_site_lon, coastline_lat,  coastline_lon)
dist_highway = calculate_distance(launch_site_lat, launch_site_lon, highway_lat,    highway_lon)
dist_railway = calculate_distance(launch_site_lat, launch_site_lon, railway_lat,    railway_lon)
dist_city    = calculate_distance(launch_site_lat, launch_site_lon, city_lat,        city_lon)

print("Distancias desde CCAFS SLC-40:")
print(f"  🌊 Costa:      {dist_coast:.2f} km")
print(f"  🛣️  Autopista:  {dist_highway:.2f} km")
print(f"  🚂 Vía férrea: {dist_railway:.2f} km")
print(f"  🏙️  Ciudad:     {dist_city:.2f} km")

In [ ]:
# Crear mapa centrado en CCAFS SLC-40 con zoom cercano
site_map3 = folium.Map(location=[launch_site_lat, launch_site_lon], zoom_start=13)

# Añadir MousePosition para referencia de coordenadas
formatter = "function(num) {return L.Util.formatNum(num, 5);}"
MousePosition(
    position='topright',
    separator=' Long: ',
    empty_string='NaN',
    lng_first=False,
    num_digits=20,
    prefix='Lat:',
    lat_formatter=formatter,
    lng_formatter=formatter,
).add_to(site_map3)

# Marcador del sitio de lanzamiento
folium.Marker(
    [launch_site_lat, launch_site_lon],
    popup=folium.Popup('<b>CCAFS SLC-40</b><br>Launch Site', max_width=200),
    icon=folium.Icon(color='blue', icon='rocket', prefix='fa')
).add_to(site_map3)

# Lista de proximidades con colores
proximities = [
    {'name': '🌊 Costa Atlántica', 'lat': coastline_lat,  'lon': coastline_lon,  'dist': dist_coast,   'color': '#1a78c2'},
    {'name': '🛣️ Autopista',        'lat': highway_lat,    'lon': highway_lon,    'dist': dist_highway, 'color': '#e67e22'},
    {'name': '🚂 Vía Férrea',        'lat': railway_lat,    'lon': railway_lon,    'dist': dist_railway, 'color': '#27ae60'},
    {'name': '🏙️ Ciudad (Melbourne)','lat': city_lat,       'lon': city_lon,       'dist': dist_city,    'color': '#8e44ad'},
]

for point in proximities:
    # Marcador con etiqueta de distancia
    folium.Marker(
        [point['lat'], point['lon']],
        popup=folium.Popup(f"<b>{point['name']}</b><br>{point['dist']:.2f} km del sitio", max_width=200),
        icon=DivIcon(
            icon_size=(140, 20),
            icon_anchor=(0, 0),
            html=f'<div style="font-size:11px; color:{point["color"]}; font-weight:bold; background:white; padding:2px; border-radius:3px;">{point["dist"]:.2f} km – {point["name"]}</div>'
        )
    ).add_to(site_map3)

    # Línea desde el sitio de lanzamiento hasta cada proximidad
    folium.PolyLine(
        locations=[[launch_site_lat, launch_site_lon], [point['lat'], point['lon']]],
        weight=2,
        color=point['color'],
        dash_array='5'
    ).add_to(site_map3)

# Mostrar el mapa — TASK 3: tomar screenshot aquí
site_map3

**📸 SCREENSHOT 3:** Captura este mapa con zoom_start=13 sobre CCAFS SLC-40.

Verás las 4 líneas de distancia (costa, autopista, vía férrea, ciudad) con sus etiquetas en km.

---
## Conclusiones del análisis geográfico

| Proximidad | Distancia desde CCAFS SLC-40 |
|---|---|
| 🌊 Costa Atlántica | ~0.90 km |
| 🛣️ Autopista más cercana | ~0.59 km |
| 🚂 Vía férrea | ~1.28 km |
| 🏙️ Ciudad (Melbourne, FL) | ~51.17 km |

**Hallazgos clave:**
- Todos los sitios de lanzamiento están ubicados en la costa (máx. 1 km del mar)
- Los sitios están cerca de vías de acceso (autopistas y ferrocarril)
- Están alejados de zonas urbanas densas (>50 km de ciudades principales)
- La proximidad al ecuador favorece los lanzamientos hacia órbitas ecuatoriales (GTO, LEO)

---
## Resumen del laboratorio

En este laboratorio utilizamos `Folium` para:

1. **TASK 1** — Visualizar los 4 sitios de lanzamiento SpaceX en un mapa global con círculos y etiquetas
2. **TASK 2** — Marcar cada lanzamiento con color verde (éxito) o rojo (fallo) usando `MarkerCluster`
3. **TASK 3** — Calcular y visualizar distancias del sitio CCAFS SLC-40 a la costa, autopista, vía férrea y ciudad más cercana usando la fórmula de Haversine

Los resultados confirman que los sitios de lanzamiento siguen patrones de ubicación muy específicos: proximidad al océano, acceso a infraestructura de transporte y distancia segura de zonas pobladas.